In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from xgboost                   import XGBClassifier
from sklearn.model_selection   import StratifiedKFold
from sklearn.metrics           import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing       import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils          import get_model_train_eval
from utils.feature_engineering import add_statistical_features, drop_highly_correlated_features

In [2]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [3]:
# zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [4]:
print(X_features.shape)

# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

(76020, 149)


In [5]:
# 상관계수 높은 feature들 삭제하기 default 0.95
X_reduced, to_drop = drop_highly_correlated_features(X_features)
X_test_reduced = X_test.drop(to_drop, axis=1)

In [7]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)

In [ ]:
# 1. HyperOpt 목적 함수 정의 (이전 99perCorr HyperOpt에서 더 개선)
# 모델 최적화: HyperOpt + Stratified K-Fold 교차검증 → 안정적인 ROC-AU

# -----------------------------

''' 개선된 코드 특징 :
    - 불균형 데이터 대응: scale_pos_weight 포함 → Recall 개선
    - 정규화 파라미터 추가: reg_alpha, reg_lambda → 과적합 방지
    - 탐색 공간 확장: gamma, min_child_weight 범위 확대
    - 교차검증 적용: Stratified K-Fold → ROC-AUC 점수 안정화
    - 탐색 횟수 증가: max_evals=100 → 더 신뢰성 있는 최적값 도출
'''
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

def objective(params):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)
    aucs = []
    
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = XGBClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=params['learning_rate'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            gamma=params['gamma'],
            min_child_weight=int(params['min_child_weight']),
            scale_pos_weight=params['scale_pos_weight'],
            reg_alpha=params['reg_alpha'],
            reg_lambda=params['reg_lambda'],
            random_state=23,
            n_jobs=-1,
            use_label_encoder=False,
            eval_metric='logloss'
        )
        
        model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_val_fold)[:, 1]
        aucs.append(roc_auc_score(y_val_fold, y_proba))
    
    mean_auc = np.mean(aucs)
    return {'loss': -mean_auc, 'status': STATUS_OK, 'auc': mean_auc}

# -----------------------------
# 2. 탐색 공간 정의
# -----------------------------
space_xgb = {
    'n_estimators': hp.quniform('n_estimators', 200, 1500, 50),
    'max_depth': hp.quniform('max_depth', 3, 12, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma': hp.uniform('gamma', 0, 10),
    'min_child_weight': hp.quniform('min_child_weight', 1, 20, 1),
    'scale_pos_weight': hp.quniform('scale_pos_weight', 1, 20, 1),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 5)
}

# -----------------------------
# 3. HyperOpt 실행
# -----------------------------
trials = Trials()
best = fmin(
    fn=objective,
    space=space_xgb,
    algo=tpe.suggest,
    max_evals=100,   # 탐색 횟수 늘려 안정성 확보
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperparameters:", best)

# -----------------------------
# 4. 최적 모델 학습 및 평가
# -----------------------------
best_params = {
    'n_estimators': int(best['n_estimators']),
    'max_depth': int(best['max_depth']),
    'learning_rate': best['learning_rate'],
    'subsample': best['subsample'],
    'colsample_bytree': best['colsample_bytree'],
    'gamma': best['gamma'],
    'min_child_weight': int(best['min_child_weight']),
    'scale_pos_weight': best['scale_pos_weight'],
    'reg_alpha': best['reg_alpha'],
    'reg_lambda': best['reg_lambda']
}

xgb_best = XGBClassifier(
    **best_params,
    random_state=23,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

# 5. 학습 및 검증 평가
get_model_train_eval(xgb_best, "XGBoost_99per_hyperoptCorr_spw_alpha_lambda", X_train, X_val, y_train, y_val)

# 100%|██████████| 100/100 [11:27<00:00,  6.87s/trial, best loss: -0.838687556338235]
# Best Hyperparameters: {
    # 'colsample_bytree': np.float64(0.7009895498566958), 
    # 'gamma': np.float64(8.04439822362272), 
    # 'learning_rate': np.float64(0.01019097677223362), 
    # 'max_depth': np.float64(5.0), 
    # 'min_child_weight': np.float64(8.0), 
    # 'n_estimators': np.float64(450.0), 
    # 'reg_alpha': np.float64(0.00021647425906756723), 
    # 'reg_lambda': np.float64(2.3459719580898137), 
    # 'scale_pos_weight': np.float64(5.0), 
    # 'subsample': np.float64(0.6896286494929481)}

# AUC: 0.8504, 정확도: 0.9296, 정밀도: 0.2429, 재현율: 0.3671, F1: 0.2923
# 오차행렬:
# [[13913   689]
#  [  381   221]]
# 실행 시간: 1.2837684154510498

  0%|          | 0/100 [00:00<?, ?trial/s, best loss=?]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:39] WARNING: C:\act

  1%|          | 1/100 [00:08<14:05,  8.54s/trial, best loss: -0.8151641844306589]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:47:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "u

  2%|▏         | 2/100 [00:32<28:18, 17.33s/trial, best loss: -0.8161234789847913]

...

 99%|█████████▉| 99/100 [11:23<00:06,  6.06s/trial, best loss: -0.838687556338235]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:58:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:58:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:59:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:59:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "u

100%|██████████| 100/100 [11:27<00:00,  6.87s/trial, best loss: -0.838687556338235]
Best Hyperparameters: {'colsample_bytree': np.float64(0.7009895498566958), 'gamma': np.float64(8.04439822362272), 'learning_rate': np.float64(0.01019097677223362), 'max_depth': np.float64(5.0), 'min_child_weight': np.float64(8.0), 'n_estimators': np.float64(450.0), 'reg_alpha': np.float64(0.00021647425906756723), 'reg_lambda': np.float64(2.3459719580898137), 'scale_pos_weight': np.float64(5.0), 'subsample': np.float64(0.6896286494929481)}


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:59:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✓ 모델 저장 완료: ../models\XGBoost_HyperOpt_Improved.pkl
  파일 크기: 0.94 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8504, 정확도: 0.9296, 정밀도: 0.2429, 재현율: 0.3671, F1: 0.2923
오차행렬:
[[13913   689]
 [  381   221]]
실행 시간: 1.2837684154510498
